This notebook attempts to load existing trained siamese backbones, add a new classification head, then attempt to tune for classification.

In [1]:
# Import helper code
import sys
sys.path.insert(1, '../')
import helpers
import torch

In [19]:
import os
dataset_path = "../split_node21_sets/"
process = "arch_seg" #"lung_seg" "crop" 
train_set = "chestxray14"
model_type = "cross_attention"

bsz = 64
resize_dim = 224
# run_index = 0
# best_epoch_to_load = 
run_index = 1
best_epoch_to_load = 70
# run_index = 4
# best_epoch_to_load = 80

run_name = f"rad_unfrz_crossattn1L8H_cosine_nosym_bsz128_{run_index}"

weights = torch.load(f'logs/subsets/{train_set}/{model_type}/{process}/{run_name}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')
# weights = torch.load('logs/subsets/chestxray14/rad_unfrz_cosine_crop_0/checkpoints/best_model.pth',map_location=device)

from torchvision import transforms
import torch
base_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.ToTensor(),
])

augment_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])
# Load with default settings
train_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "train"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=augment_transform,
    cache_in_ram=True

)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  2.85it/s]


Caching base images into RAM...


Caching: 100%|██████████| 2478/2478 [01:14<00:00, 33.42it/s]

Cached 2478 images
Total pairs: 1239
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 420
  normal (idx=0): 819


In [4]:
test_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,train_set, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  9.95it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1062/1062 [00:06<00:00, 152.90it/s]

Cached 1062 images
Total pairs: 531
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 180
  normal (idx=0): 351


In [5]:
test2_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,"padchest", "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  6.17it/s]


Caching base images into RAM...


Caching: 100%|██████████| 1008/1008 [00:06<00:00, 154.32it/s]

Cached 1008 images
Total pairs: 504
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 94
  normal (idx=0): 410


In [6]:
test3_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process,"jsrt", "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 40.16it/s]


Caching base images into RAM...


Caching: 100%|██████████| 144/144 [00:00<00:00, 146.86it/s]

Cached 144 images
Total pairs: 72
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 44
  normal (idx=0): 28


## Create Classifier Head

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class CrossAttentionClassifier(nn.Module):
    """
    Wraps trained CrossAttentionSiamese with a classification head
    Default freeze the cross-attention encoder for fast fine-tuning
    Matches SiameseClassifier architecture with distance feature
    """
    def __init__(self, crossattn_model, embedding_dim=128, freeze_encoder=True):
        super(CrossAttentionClassifier, self).__init__()
        
        self.encoder = crossattn_model
        
        # Freeze the encoder if desired
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        
        # Classification head operates on distance + concatenated embeddings
        input_dim = 2 * embedding_dim + 1  # concatenated embeddings + distance
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)  # Binary classification (sigmoid applied later)
        )
        
    def forward(self, x1, x2, return_embeddings=False, distance_metric='cosine'):
        """
        Args:
            x1, x2: Input image pairs
            return_embeddings: If True, also return embeddings for analysis
            distance_metric: 'euclidean' or 'cosine'
        Returns:
            logits: Classification logits (before sigmoid)
            (optional) emb1, emb2, distance
        """
        # Get embeddings from cross-attention network
        emb1, emb2 = self.encoder(x1, x2)
        
        # Calculate distance
        if distance_metric == 'euclidean':
            distance = F.pairwise_distance(emb1, emb2, p=2)
        elif distance_metric == 'cosine':
            cosine_sim = torch.sum(emb1 * emb2, dim=1)
            distance = 1 - cosine_sim
        else:
            raise ValueError(f"Unknown distance metric: {distance_metric}")
        
        # Concatenate embeddings and distance
        # Shape: (batch, 2*embedding_dim + 1)
        combined = torch.cat([emb1, emb2, distance.unsqueeze(1)], dim=1)
        
        # Get classification logits
        logits = self.classifier(combined).squeeze()
        
        if return_embeddings:
            return logits, emb1, emb2, distance
        return logits

In [9]:
# import training metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

In [11]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [12]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5629 | Acc: 0.8402 | Prec: 0.9664 | Rec: 0.5476 | F1: 0.6991 | AUC: 0.9309
  Test   Loss: 0.4702 | Acc: 0.8249 | Prec: 0.7736 | Rec: 0.6833 | F1: 0.7257 | AUC: 0.8211
  Padchest   Loss: 0.4447 | Acc: 0.8175 | Prec: 0.5114 | Rec: 0.4787 | F1: 0.4945 | AUC: 0.7572
  JSRT   Loss: 1.0637 | Acc: 0.4722 | Prec: 0.8000 | Rec: 0.1818 | F1: 0.2963 | AUC: 0.5430
  ✓ Saved new best model! (F1: 0.7257)
Epoch 2/20
  Train Loss: 0.2259 | Acc: 0.9540 | Prec: 0.9666 | Rec: 0.8952 | F1: 0.9295 | AUC: 0.9621
  Test   Loss: 0.5759 | Acc: 0.8211 | Prec: 0.7485 | Rec: 0.7111 | F1: 0.7293 | AUC: 0.8251
  Padchest   Loss: 0.6286 | Acc: 0.7877 | Prec: 0.4414 | Rec: 0.5213 | F1: 0.4780 | AUC: 0.7610
  JSRT   Loss: 1.3300 | Acc: 0.4583 | Prec: 0.6923 | Rec: 0.2045 | F1: 0.3158 | AUC: 0.5731
  ✓ Saved new best model! (F1: 0.7293)
Epoch 3/20
  Train Loss: 0.1546 | Acc: 0.9548 | Prec: 0.9333 | Rec: 0.9333 | F1: 0.9333 | AUC: 0.9728
  Test   Loss: 0.6768 | Acc: 0.8136 | Prec: 0.7166 | Rec

In [15]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5687 | Acc: 0.8733 | Prec: 0.7757 | Rec: 0.8810 | F1: 0.8250 | AUC: 0.9402
  Test   Loss: 0.4777 | Acc: 0.8211 | Prec: 0.7640 | Rec: 0.6833 | F1: 0.7214 | AUC: 0.8164
  Padchest   Loss: 0.4644 | Acc: 0.8095 | Prec: 0.4894 | Rec: 0.4894 | F1: 0.4894 | AUC: 0.7549
  JSRT   Loss: 0.9274 | Acc: 0.4444 | Prec: 0.6667 | Rec: 0.1818 | F1: 0.2857 | AUC: 0.5300
  ✓ Saved new best model! (F1: 0.7214)
Epoch 2/20
  Train Loss: 0.2193 | Acc: 0.9524 | Prec: 0.9593 | Rec: 0.8976 | F1: 0.9274 | AUC: 0.9711
  Test   Loss: 0.6050 | Acc: 0.8173 | Prec: 0.7427 | Rec: 0.7056 | F1: 0.7236 | AUC: 0.8172
  Padchest   Loss: 0.6524 | Acc: 0.7897 | Prec: 0.4455 | Rec: 0.5213 | F1: 0.4804 | AUC: 0.7528
  JSRT   Loss: 1.9329 | Acc: 0.4583 | Prec: 0.6923 | Rec: 0.2045 | F1: 0.3158 | AUC: 0.5414
  ✓ Saved new best model! (F1: 0.7236)
Epoch 3/20
  Train Loss: 0.1561 | Acc: 0.9532 | Prec: 0.9393 | Rec: 0.9214 | F1: 0.9303 | AUC: 0.9727
  Test   Loss: 0.7622 | Acc: 0.7966 | Prec: 0.6875 | Rec

In [ ]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

### Model 1

In [11]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [12]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5629 | Acc: 0.8402 | Prec: 0.9664 | Rec: 0.5476 | F1: 0.6991 | AUC: 0.9309
  Test   Loss: 0.4702 | Acc: 0.8249 | Prec: 0.7736 | Rec: 0.6833 | F1: 0.7257 | AUC: 0.8211
  Padchest   Loss: 0.4447 | Acc: 0.8175 | Prec: 0.5114 | Rec: 0.4787 | F1: 0.4945 | AUC: 0.7572
  JSRT   Loss: 1.0637 | Acc: 0.4722 | Prec: 0.8000 | Rec: 0.1818 | F1: 0.2963 | AUC: 0.5430
  ✓ Saved new best model! (F1: 0.7257)
Epoch 2/20
  Train Loss: 0.2259 | Acc: 0.9540 | Prec: 0.9666 | Rec: 0.8952 | F1: 0.9295 | AUC: 0.9621
  Test   Loss: 0.5759 | Acc: 0.8211 | Prec: 0.7485 | Rec: 0.7111 | F1: 0.7293 | AUC: 0.8251
  Padchest   Loss: 0.6286 | Acc: 0.7877 | Prec: 0.4414 | Rec: 0.5213 | F1: 0.4780 | AUC: 0.7610
  JSRT   Loss: 1.3300 | Acc: 0.4583 | Prec: 0.6923 | Rec: 0.2045 | F1: 0.3158 | AUC: 0.5731
  ✓ Saved new best model! (F1: 0.7293)
Epoch 3/20
  Train Loss: 0.1546 | Acc: 0.9548 | Prec: 0.9333 | Rec: 0.9333 | F1: 0.9333 | AUC: 0.9728
  Test   Loss: 0.6768 | Acc: 0.8136 | Prec: 0.7166 | Rec

In [15]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

In [16]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5687 | Acc: 0.8733 | Prec: 0.7757 | Rec: 0.8810 | F1: 0.8250 | AUC: 0.9402
  Test   Loss: 0.4777 | Acc: 0.8211 | Prec: 0.7640 | Rec: 0.6833 | F1: 0.7214 | AUC: 0.8164
  Padchest   Loss: 0.4644 | Acc: 0.8095 | Prec: 0.4894 | Rec: 0.4894 | F1: 0.4894 | AUC: 0.7549
  JSRT   Loss: 0.9274 | Acc: 0.4444 | Prec: 0.6667 | Rec: 0.1818 | F1: 0.2857 | AUC: 0.5300
  ✓ Saved new best model! (F1: 0.7214)
Epoch 2/20
  Train Loss: 0.2193 | Acc: 0.9524 | Prec: 0.9593 | Rec: 0.8976 | F1: 0.9274 | AUC: 0.9711
  Test   Loss: 0.6050 | Acc: 0.8173 | Prec: 0.7427 | Rec: 0.7056 | F1: 0.7236 | AUC: 0.8172
  Padchest   Loss: 0.6524 | Acc: 0.7897 | Prec: 0.4455 | Rec: 0.5213 | F1: 0.4804 | AUC: 0.7528
  JSRT   Loss: 1.9329 | Acc: 0.4583 | Prec: 0.6923 | Rec: 0.2045 | F1: 0.3158 | AUC: 0.5414
  ✓ Saved new best model! (F1: 0.7236)
Epoch 3/20
  Train Loss: 0.1561 | Acc: 0.9532 | Prec: 0.9393 | Rec: 0.9214 | F1: 0.9303 | AUC: 0.9727
  Test   Loss: 0.7622 | Acc: 0.7966 | Prec: 0.6875 | Rec

In [17]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [18]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5897 | Acc: 0.8797 | Prec: 0.8379 | Rec: 0.8000 | F1: 0.8185 | AUC: 0.9327
  Test   Loss: 0.4935 | Acc: 0.8286 | Prec: 0.7730 | Rec: 0.7000 | F1: 0.7347 | AUC: 0.8210
  Padchest   Loss: 0.4803 | Acc: 0.8175 | Prec: 0.5109 | Rec: 0.5000 | F1: 0.5054 | AUC: 0.7500
  JSRT   Loss: 0.8550 | Acc: 0.4722 | Prec: 0.8000 | Rec: 0.1818 | F1: 0.2963 | AUC: 0.5203
  ✓ Saved new best model! (F1: 0.7347)
Epoch 2/20
  Train Loss: 0.2389 | Acc: 0.9572 | Prec: 0.9693 | Rec: 0.9024 | F1: 0.9346 | AUC: 0.9707
  Test   Loss: 0.5434 | Acc: 0.8060 | Prec: 0.7059 | Rec: 0.7333 | F1: 0.7193 | AUC: 0.8323
  Padchest   Loss: 0.6687 | Acc: 0.7758 | Prec: 0.4252 | Rec: 0.5745 | F1: 0.4887 | AUC: 0.7563
  JSRT   Loss: 1.5886 | Acc: 0.5000 | Prec: 0.7500 | Rec: 0.2727 | F1: 0.4000 | AUC: 0.6047
Epoch 3/20
  Train Loss: 0.1506 | Acc: 0.9516 | Prec: 0.9265 | Rec: 0.9310 | F1: 0.9287 | AUC: 0.9746
  Test   Loss: 0.6510 | Acc: 0.8117 | Prec: 0.7198 | Rec: 0.7278 | F1: 0.7238 | AUC: 0.8384
  P

### Model 2

In [20]:
import os
bsz = 64
resize_dim = 224
run_index = 4
best_epoch_to_load = 80

run_name = f"rad_unfrz_crossattn1L8H_cosine_nosym_bsz128_{run_index}"

weights = torch.load(f'logs/subsets/{train_set}/{model_type}/{process}/{run_name}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')


In [21]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [22]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5736 | Acc: 0.8321 | Prec: 0.9907 | Rec: 0.5095 | F1: 0.6730 | AUC: 0.9762
  Test   Loss: 0.5182 | Acc: 0.7947 | Prec: 0.8087 | Rec: 0.5167 | F1: 0.6305 | AUC: 0.8323
  Padchest   Loss: 0.4985 | Acc: 0.8194 | Prec: 0.5165 | Rec: 0.5000 | F1: 0.5081 | AUC: 0.7743
  JSRT   Loss: 0.8044 | Acc: 0.4861 | Prec: 0.8182 | Rec: 0.2045 | F1: 0.3273 | AUC: 0.6713
  ✓ Saved new best model! (F1: 0.6305)
Epoch 2/20
  Train Loss: 0.2218 | Acc: 0.9742 | Prec: 0.9755 | Rec: 0.9476 | F1: 0.9614 | AUC: 0.9878
  Test   Loss: 0.6738 | Acc: 0.7947 | Prec: 0.7983 | Rec: 0.5278 | F1: 0.6355 | AUC: 0.8335
  Padchest   Loss: 0.6625 | Acc: 0.8175 | Prec: 0.5106 | Rec: 0.5106 | F1: 0.5106 | AUC: 0.7731
  JSRT   Loss: 1.5841 | Acc: 0.5000 | Prec: 0.9000 | Rec: 0.2045 | F1: 0.3333 | AUC: 0.6778
  ✓ Saved new best model! (F1: 0.6355)
Epoch 3/20
  Train Loss: 0.1033 | Acc: 0.9726 | Prec: 0.9754 | Rec: 0.9429 | F1: 0.9588 | AUC: 0.9852
  Test   Loss: 0.7791 | Acc: 0.8023 | Prec: 0.8049 | Rec

In [23]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [24]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5475 | Acc: 0.7700 | Prec: 0.9787 | Rec: 0.3286 | F1: 0.4920 | AUC: 0.9608
  Test   Loss: 0.4977 | Acc: 0.7947 | Prec: 0.8257 | Rec: 0.5000 | F1: 0.6228 | AUC: 0.8343
  Padchest   Loss: 0.4300 | Acc: 0.8294 | Prec: 0.5476 | Rec: 0.4894 | F1: 0.5169 | AUC: 0.7764
  JSRT   Loss: 0.9596 | Acc: 0.5000 | Prec: 0.9000 | Rec: 0.2045 | F1: 0.3333 | AUC: 0.6542
  ✓ Saved new best model! (F1: 0.6228)
Epoch 2/20
  Train Loss: 0.1984 | Acc: 0.9653 | Prec: 0.9896 | Rec: 0.9071 | F1: 0.9466 | AUC: 0.9824
  Test   Loss: 0.7497 | Acc: 0.8023 | Prec: 0.8205 | Rec: 0.5333 | F1: 0.6465 | AUC: 0.7796
  Padchest   Loss: 0.6711 | Acc: 0.8155 | Prec: 0.5054 | Rec: 0.5000 | F1: 0.5027 | AUC: 0.7460
  JSRT   Loss: 1.9891 | Acc: 0.4861 | Prec: 0.8182 | Rec: 0.2045 | F1: 0.3273 | AUC: 0.6737
  ✓ Saved new best model! (F1: 0.6465)
Epoch 3/20
  Train Loss: 0.0912 | Acc: 0.9750 | Prec: 0.9802 | Rec: 0.9452 | F1: 0.9624 | AUC: 0.9889
  Test   Loss: 0.8606 | Acc: 0.7947 | Prec: 0.7886 | Rec

In [25]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [26]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5941 | Acc: 0.8709 | Prec: 0.7559 | Rec: 0.9143 | F1: 0.8276 | AUC: 0.9492
  Test   Loss: 0.5135 | Acc: 0.8004 | Prec: 0.8491 | Rec: 0.5000 | F1: 0.6294 | AUC: 0.8357
  Padchest   Loss: 0.4763 | Acc: 0.8254 | Prec: 0.5349 | Rec: 0.4894 | F1: 0.5111 | AUC: 0.7785
  JSRT   Loss: 0.8246 | Acc: 0.5000 | Prec: 0.9000 | Rec: 0.2045 | F1: 0.3333 | AUC: 0.6640
  ✓ Saved new best model! (F1: 0.6294)
Epoch 2/20
  Train Loss: 0.2113 | Acc: 0.9669 | Prec: 0.9773 | Rec: 0.9238 | F1: 0.9498 | AUC: 0.9826
  Test   Loss: 0.7372 | Acc: 0.7928 | Prec: 0.7778 | Rec: 0.5444 | F1: 0.6405 | AUC: 0.7955
  Padchest   Loss: 0.6848 | Acc: 0.8155 | Prec: 0.5052 | Rec: 0.5213 | F1: 0.5131 | AUC: 0.7446
  JSRT   Loss: 1.8371 | Acc: 0.4861 | Prec: 0.7692 | Rec: 0.2273 | F1: 0.3509 | AUC: 0.6956
  ✓ Saved new best model! (F1: 0.6405)
Epoch 3/20
  Train Loss: 0.1025 | Acc: 0.9718 | Prec: 0.9923 | Rec: 0.9238 | F1: 0.9568 | AUC: 0.9849
  Test   Loss: 0.7963 | Acc: 0.8117 | Prec: 0.7778 | Rec

### Model 3

In [54]:
import os
bsz = 64
resize_dim = 224
run_index = 5
# best_epoch_to_load = 90 # ONLY PRED 0.33
best_epoch_to_load = 60

run_name = f"rad_unfrz_crossattn1L8H_cosine_nosym_bsz128_{run_index}"

weights = torch.load(f'logs/subsets/{train_set}/{model_type}/{process}/{run_name}/checkpoints/checkpoint_epoch_{best_epoch_to_load}.pth')


In [55]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)

model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [56]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_1.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.6067 | Acc: 0.7837 | Prec: 0.6490 | Rec: 0.7881 | F1: 0.7118 | AUC: 0.8445
  Test   Loss: 0.5155 | Acc: 0.8041 | Prec: 0.8958 | Rec: 0.4778 | F1: 0.6232 | AUC: 0.8834
  Padchest   Loss: 0.4374 | Acc: 0.8591 | Prec: 0.7949 | Rec: 0.3298 | F1: 0.4662 | AUC: 0.8132
  JSRT   Loss: 0.9379 | Acc: 0.4306 | Prec: 1.0000 | Rec: 0.0682 | F1: 0.1277 | AUC: 0.6104
  ✓ Saved new best model! (F1: 0.6232)
Epoch 2/20
  Train Loss: 0.3064 | Acc: 0.9241 | Prec: 0.9711 | Rec: 0.8000 | F1: 0.8773 | AUC: 0.9443
  Test   Loss: 0.4870 | Acc: 0.8154 | Prec: 0.8596 | Rec: 0.5444 | F1: 0.6667 | AUC: 0.8821
  Padchest   Loss: 0.3957 | Acc: 0.8532 | Prec: 0.7000 | Rec: 0.3723 | F1: 0.4861 | AUC: 0.8140
  JSRT   Loss: 1.0886 | Acc: 0.4583 | Prec: 1.0000 | Rec: 0.1136 | F1: 0.2041 | AUC: 0.6153
  ✓ Saved new best model! (F1: 0.6667)
Epoch 3/20
  Train Loss: 0.2256 | Acc: 0.9290 | Prec: 0.9323 | Rec: 0.8524 | F1: 0.8905 | AUC: 0.9435
  Test   Loss: 0.5671 | Acc: 0.8136 | Prec: 0.8584 | Rec

In [57]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [58]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_2.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.6113 | Acc: 0.8257 | Prec: 0.7040 | Rec: 0.8381 | F1: 0.7652 | AUC: 0.8970
  Test   Loss: 0.5167 | Acc: 0.8041 | Prec: 0.8725 | Rec: 0.4944 | F1: 0.6312 | AUC: 0.8827
  Padchest   Loss: 0.4580 | Acc: 0.8611 | Prec: 0.7727 | Rec: 0.3617 | F1: 0.4928 | AUC: 0.8186
  JSRT   Loss: 0.9335 | Acc: 0.4306 | Prec: 1.0000 | Rec: 0.0682 | F1: 0.1277 | AUC: 0.6006
  ✓ Saved new best model! (F1: 0.6312)
Epoch 2/20
  Train Loss: 0.3064 | Acc: 0.9274 | Prec: 0.9741 | Rec: 0.8071 | F1: 0.8828 | AUC: 0.9534
  Test   Loss: 0.5341 | Acc: 0.8154 | Prec: 0.8661 | Rec: 0.5389 | F1: 0.6644 | AUC: 0.8833
  Padchest   Loss: 0.4313 | Acc: 0.8591 | Prec: 0.7347 | Rec: 0.3830 | F1: 0.5035 | AUC: 0.8157
  JSRT   Loss: 1.4326 | Acc: 0.4583 | Prec: 1.0000 | Rec: 0.1136 | F1: 0.2041 | AUC: 0.6161
  ✓ Saved new best model! (F1: 0.6644)
Epoch 3/20
  Train Loss: 0.2000 | Acc: 0.9427 | Prec: 0.9509 | Rec: 0.8762 | F1: 0.9120 | AUC: 0.9521
  Test   Loss: 0.5601 | Acc: 0.8173 | Prec: 0.8609 | Rec

In [59]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
#                        embedding_dim=128, freeze_backbone=True).to(device)


model = helpers.crossattention.CrossAttentionSiamese(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, num_attn_layers=1, num_heads=8, freeze_backbone=True).to(device)

model.load_state_dict(weights['model_state_dict'])

classifier = CrossAttentionClassifier(model, embedding_dim=128, freeze_encoder=True)

optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, classifier.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
)

/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [ ]:
num_epochs = 20
best_test_f1 = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # Training
    classifier.train()
    classifier.to(device)
    train_loss = 0.0
    train_preds = []
    train_labels_list = []

    for _, (x1, x2, labels, _path1, _path2) in enumerate(train_dataloader):
        x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        logits = classifier(x1, x2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.sigmoid(logits)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())
    
    train_loss /= len(train_dataloader)
    # calculate metrics
    train_preds_binary = (np.array(train_preds) > 0.5).astype(int)
    train_labels_array = np.array(train_labels_list)
    
    train_acc = accuracy_score(train_labels_array, train_preds_binary)
    train_prec = precision_score(train_labels_array, train_preds_binary, zero_division=0)
    train_rec = recall_score(train_labels_array, train_preds_binary, zero_division=0)
    train_f1 = f1_score(train_labels_array, train_preds_binary, zero_division=0)
    train_auc = roc_auc_score(train_labels_array, train_preds) if len(np.unique(train_labels_array)) > 1 else 0.0
    
    
    # test
    classifier.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_loss /= len(test_dataloader)
    # Calculate validation metrics
    val_preds_binary = (np.array(all_preds) > 0.5).astype(int)
    val_labels_array = np.array(all_labels)
    
    val_acc = accuracy_score(val_labels_array, val_preds_binary)
    val_prec = precision_score(val_labels_array, val_preds_binary, zero_division=0)
    val_rec = recall_score(val_labels_array, val_preds_binary, zero_division=0)
    val_f1 = f1_score(val_labels_array, val_preds_binary, zero_division=0)
    val_auc = roc_auc_score(val_labels_array, all_preds) if len(np.unique(val_labels_array)) > 1 else 0.0
    scheduler.step(test_loss)

    test2_loss = 0.0
    all2_preds = []
    all2_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test2_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test2_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all2_preds.extend(preds.cpu().numpy())
            all2_labels.extend(labels.cpu().numpy())
    
    test2_loss /= len(test2_dataloader)
    # Calculate validation metrics
    val2_preds_binary = (np.array(all2_preds) > 0.5).astype(int)
    val2_labels_array = np.array(all2_labels)
    
    val2_acc = accuracy_score(val2_labels_array, val2_preds_binary)
    val2_prec = precision_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_rec = recall_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_f1 = f1_score(val2_labels_array, val2_preds_binary, zero_division=0)
    val2_auc = roc_auc_score(val2_labels_array, all2_preds) if len(np.unique(val2_labels_array)) > 1 else 0.0
    scheduler.step(test2_loss)

    test3_loss = 0.0
    all3_preds = []
    all3_labels = []
    
    with torch.no_grad():
        for _, (x1, x2, labels, _path1, _path2) in enumerate(test3_dataloader):
            x1, x2, labels = x1.to(device), x2.to(device), labels.float().to(device)
            
            logits = classifier(x1, x2)
            loss = criterion(logits, labels)
            test3_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all3_preds.extend(preds.cpu().numpy())
            all3_labels.extend(labels.cpu().numpy())
    
    test3_loss /= len(test3_dataloader)
    # Calculate validation metrics
    val3_preds_binary = (np.array(all3_preds) > 0.5).astype(int)
    val3_labels_array = np.array(all3_labels)
    
    val3_acc = accuracy_score(val3_labels_array, val3_preds_binary)
    val3_prec = precision_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_rec = recall_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_f1 = f1_score(val3_labels_array, val3_preds_binary, zero_division=0)
    val3_auc = roc_auc_score(val3_labels_array, all3_preds) if len(np.unique(val3_labels_array)) > 1 else 0.0
    scheduler.step(test3_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print(f"  Test   Loss: {test_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  Padchest   Loss: {test2_loss:.4f} | Acc: {val2_acc:.4f} | Prec: {val2_prec:.4f} | Rec: {val2_rec:.4f} | F1: {val2_f1:.4f} | AUC: {val2_auc:.4f}")
    print(f"  JSRT   Loss: {test3_loss:.4f} | Acc: {val3_acc:.4f} | Prec: {val3_prec:.4f} | Rec: {val3_rec:.4f} | F1: {val3_f1:.4f} | AUC: {val3_auc:.4f}")

    # Save best model based on Test F1
    if val_f1 > best_test_f1:
        best_test_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(classifier.state_dict(), f'cxr14_heads/GlobalAttention/{run_name}_{run_index}_best_classifier_f1_3.pth')
        print(f"  ✓ Saved new best model! (F1: {val_f1:.4f})")

print(f"\nTraining complete! Best model from Epoch {best_epoch} with Test F1: {best_test_f1:.4f}")

Epoch 1/20
  Train Loss: 0.5905 | Acc: 0.7635 | Prec: 0.6427 | Rec: 0.6810 | F1: 0.6613 | AUC: 0.8298
  Test   Loss: 0.5019 | Acc: 0.8004 | Prec: 0.8936 | Rec: 0.4667 | F1: 0.6131 | AUC: 0.8860
  Padchest   Loss: 0.4149 | Acc: 0.8611 | Prec: 0.8333 | Rec: 0.3191 | F1: 0.4615 | AUC: 0.8140
  JSRT   Loss: 1.0508 | Acc: 0.4306 | Prec: 1.0000 | Rec: 0.0682 | F1: 0.1277 | AUC: 0.5917
  ✓ Saved new best model! (F1: 0.6131)
Epoch 2/20
  Train Loss: 0.2868 | Acc: 0.9257 | Prec: 0.9659 | Rec: 0.8095 | F1: 0.8808 | AUC: 0.9478
  Test   Loss: 0.5387 | Acc: 0.8136 | Prec: 0.8584 | Rec: 0.5389 | F1: 0.6621 | AUC: 0.8809
  Padchest   Loss: 0.4126 | Acc: 0.8611 | Prec: 0.7609 | Rec: 0.3723 | F1: 0.5000 | AUC: 0.8115
  JSRT   Loss: 1.5956 | Acc: 0.4722 | Prec: 1.0000 | Rec: 0.1364 | F1: 0.2400 | AUC: 0.6136
  ✓ Saved new best model! (F1: 0.6621)
Epoch 3/20
  Train Loss: 0.2256 | Acc: 0.9346 | Prec: 0.9357 | Rec: 0.8667 | F1: 0.8999 | AUC: 0.9546
  Test   Loss: 0.5460 | Acc: 0.8192 | Prec: 0.8621 | Rec